In [1]:
import pandas as pd

path = "mockdata/"

df_waterings = pd.read_csv(f'{path}waterings.csv')
df_waterings = df_waterings.drop(columns=['PredictedFutureWaterTime'])
df_sensors = pd.read_csv(f'{path}sensor_datas.csv')
df_plants = pd.read_csv(f'{path}plants.csv')

df_sensors

,PlantMAC,Temperature,AirHumidity,SoilHumidity,LightIntensity,Timestamp
0,34:2c:d8:10:0f:2f,20.29,53.14,57.95,608.3,2025-01-01T12:00:00
1,34:2c:d8:10:0f:2f,19.28,51.92,58.10,558.1,2025-01-01T13:00:00
2,34:2c:d8:10:0f:2f,19.27,52.32,57.64,676.7,2025-01-01T14:00:00
3,34:2c:d8:10:0f:2f,19.04,52.66,57.52,674.2,2025-01-01T15:00:00
4,34:2c:d8:10:0f:2f,20.14,50.51,57.74,510.5,2025-01-01T16:00:00
...,...,...,...,...,...,...
149995,32:e4:91:31:44:e8,20.33,69.23,54.95,12.6,2026-02-21T23:00:00
149996,32:e4:91:31:44:e8,21.44,71.48,55.15,0.0,2026-02-22T00:00:00
149997,32:e4:91:31:44:e8,NaN,67.97,54.93,25.2,2026-02-22T01:00:00
149998,32:e4:91:31:44:e8,20.29,69.54,54.67,42.0,2026-02-22T02:00:00


In [2]:
#1 MAC Mapping
mac_map = {mac: i for i, mac in enumerate(df_plants['MAC'].unique())}

#2 Prepare waterings

df_waterings = df_waterings.rename(columns={'LastWaterTime': 'WaterTimestamp'})

#3 Convert timestamps
df_sensors['Timestamp']   = pd.to_datetime(df_sensors['Timestamp'])
df_waterings['WaterTimestamp'] = pd.to_datetime(df_waterings['WaterTimestamp'], format='mixed')

#4 Add mac_id to all dataframes
df_sensors['mac_id']   = df_sensors['PlantMAC'].map(mac_map)
df_waterings['mac_id'] = df_waterings['PlantMAC'].map(mac_map)
df_plants['mac_id']    = df_plants['MAC'].map(mac_map)

#5 Clean plants table
df_plants_cleaned = df_plants.drop(columns=['MAC', 'Username', 'Name'])

sensors_sorted = df_sensors.sort_values(['Timestamp']).reset_index(drop=True)
waterings_sorted = df_waterings.sort_values(['WaterTimestamp']).reset_index(drop=True)

sensors_sorted

,PlantMAC,Temperature,AirHumidity,SoilHumidity,LightIntensity,Timestamp,mac_id
0,34:2c:d8:10:0f:2f,20.29,53.14,57.95,608.3,2025-01-01 12:00:00,0
1,fd:68:83:89:b4:a6,18.61,53.53,58.18,NaN,2025-01-01 12:00:00,12
2,94:64:cb:7c:b1:16,20.25,52.04,57.32,690.0,2025-01-01 12:00:00,10
3,c9:fe:02:2a:7a:ba,25.32,21.03,17.62,968.2,2025-01-01 12:00:00,5
4,34:8e:15:80:87:53,18.54,54.54,57.99,716.2,2025-01-01 12:00:00,6
...,...,...,...,...,...,...,...
149995,7c:d9:2c:2b:b3:14,20.53,51.44,36.46,NaN,2026-02-22 03:00:00,1
149996,fd:68:83:89:b4:a6,16.53,55.50,46.81,0.0,2026-02-22 03:00:00,12
149997,34:2c:d8:10:0f:2f,19.73,53.48,46.38,8.5,2026-02-22 03:00:00,0
149998,6b:85:cf:f3:7b:bb,18.50,44.04,24.13,0.0,2026-02-22 03:00:00,9


In [3]:
sensors_sorted.isnull().sum()

PlantMAC              0
Temperature       10528
AirHumidity       10735
SoilHumidity      10297
LightIntensity    10569
Timestamp             0
mac_id                0
dtype: int64

In [4]:
missing_percent = sensors_sorted.isnull().mean() * 100
print(missing_percent)

PlantMAC          0.000000
Temperature       7.018667
AirHumidity       7.156667
SoilHumidity      6.864667
LightIntensity    7.046000
Timestamp         0.000000
mac_id            0.000000
dtype: float64


In [5]:
df_water_sensors = pd.merge_asof(
    sensors_sorted,
    waterings_sorted[['mac_id', 'WaterTimestamp', 'PumpTimeInSeconds', 'WaterLevel']],
    left_on='Timestamp',
    right_on='WaterTimestamp',
    by='mac_id',
    direction='backward',
    tolerance=pd.Timedelta(days=90)
)

df_water_sensors['hours_since_watering'] = (
    df_water_sensors['Timestamp'] - df_water_sensors['WaterTimestamp']
).dt.total_seconds() /3600

# Readings that surpass the plant being watered for 90 days are removed
df_water_sensors = df_water_sensors[~df_water_sensors['hours_since_watering'].isnull()]

df_water_sensors = df_water_sensors.drop(columns=['WaterTimestamp', 'Timestamp'])


# Bring in plant optimal values
df = df_water_sensors.merge(df_plants_cleaned, on='mac_id', how='left')

# Remove impossible outliers
## Team decided the following sensor values are highly like errors above 9000 light,
# above 50C and below 5C, soil humidity above 100 or below 0, air humidity below 5 and above 95

df = df[(df["SoilHumidity"] > 0) & (df["SoilHumidity"]<100) | (df["SoilHumidity"].isnull())]
df = df[(df["Temperature"] > 5) & (df["Temperature"]<50) | (df["Temperature"].isnull())]
df = df[(df["AirHumidity"] > 5) & (df["AirHumidity"]<95) | (df["AirHumidity"].isnull())]
df = df[(df["LightIntensity"] > 0) & (df["LightIntensity"]<9000) | (df["LightIntensity"].isnull())]

df

,PlantMAC,Temperature,AirHumidity,SoilHumidity,LightIntensity,mac_id,PumpTimeInSeconds,WaterLevel,hours_since_watering,OptimalTemperature,OptimalAirHumidity,OptimalSoilHumidity,OptimalLightIntensity
0,34:2c:d8:10:0f:2f,20.29,53.14,57.95,608.3,0,0.00,89.35,0.0,20.7,50.8,57.5,537.1
1,fd:68:83:89:b4:a6,18.61,53.53,58.18,NaN,12,0.00,86.83,0.0,19.3,51.3,58.5,549.4
2,94:64:cb:7c:b1:16,20.25,52.04,57.32,690.0,10,0.00,86.12,0.0,19.8,51.6,57.5,663.0
3,c9:fe:02:2a:7a:ba,25.32,21.03,17.62,968.2,5,0.00,99.02,0.0,26.7,19.3,17.3,846.4
4,34:8e:15:80:87:53,18.54,54.54,57.99,716.2,6,0.00,90.73,0.0,19.2,50.8,58.1,654.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...
134271,1b:4c:5b:4c:43:9d,20.05,31.31,9.00,NaN,13,5.65,40.73,170.0,21.8,29.2,25.1,772.5
134273,94:64:cb:7c:b1:16,17.81,53.49,50.85,26.3,10,5.46,81.27,363.0,19.8,51.6,57.5,663.0
134276,cd:86:ad:9d:4e:9f,19.81,39.91,25.46,17.4,11,7.21,56.42,156.0,21.6,39.6,40.7,447.8
134277,7c:d9:2c:2b:b3:14,20.53,51.44,36.46,NaN,1,6.71,23.52,21.0,21.8,50.9,50.3,456.6


In [6]:
# Deviation function

df['soil_deficit']= df['OptimalSoilHumidity'] - df['SoilHumidity']
df['soil_deficit_ratio'] = df['soil_deficit'] /df['OptimalSoilHumidity']

df['temp_deviation'] = df['Temperature'] - df['OptimalTemperature']
df['air_hum_deficit'] = df['OptimalAirHumidity'] - df['AirHumidity']
df['light_deviation'] = df['LightIntensity'] - df['OptimalLightIntensity']

# Interaction features
df['deficit_x_temp'] = df['soil_deficit'] * df['temp_deviation']
df['deficit_x_light'] = df['soil_deficit'] * df['light_deviation']
df['deficit_x_air'] = df['soil_deficit'] * df['air_hum_deficit']


# Estimated evaporation
df['et_approx'] = (df['Temperature'] * df['LightIntensity']) / (df['AirHumidity'] + 1)

df = df.drop(columns=['PlantMAC','Temperature','LightIntensity','AirHumidity','SoilHumidity','OptimalAirHumidity','OptimalSoilHumidity','OptimalLightIntensity','OptimalTemperature'])
df

,mac_id,PumpTimeInSeconds,WaterLevel,hours_since_watering,soil_deficit,soil_deficit_ratio,temp_deviation,air_hum_deficit,light_deviation,deficit_x_temp,deficit_x_light,deficit_x_air,et_approx
0,0,0.00,89.35,0.0,-0.45,-0.007826,-0.41,-2.34,71.2,0.1845,-32.040,1.0530,227.972054
1,12,0.00,86.83,0.0,0.32,0.005470,-0.69,-2.23,NaN,-0.2208,NaN,-0.7136,NaN
2,10,0.00,86.12,0.0,0.18,0.003130,0.45,-0.44,27.0,0.0810,4.860,-0.0792,263.433258
3,5,0.00,99.02,0.0,-0.32,-0.018497,-1.38,-1.73,121.8,0.4416,-38.976,0.5536,1112.792737
4,6,0.00,90.73,0.0,0.11,0.001893,-0.66,-3.74,61.3,-0.0726,6.743,-0.4114,239.077206
...,...,...,...,...,...,...,...,...,...,...,...,...,...
134271,13,5.65,40.73,170.0,16.10,0.641434,-1.75,-2.11,NaN,-28.1750,NaN,-33.9710,NaN
134273,10,5.46,81.27,363.0,6.65,0.115652,-1.99,-1.89,-636.7,-13.2335,-4234.055,-12.5685,8.596128
134276,11,7.21,56.42,156.0,15.24,0.374447,-1.79,-0.31,-430.4,-27.2796,-6559.296,-4.7244,8.425666
134277,1,6.71,23.52,21.0,13.84,0.275149,-1.27,-0.54,NaN,-17.5768,NaN,-7.4736,NaN


In [7]:
df.isnull().sum()

mac_id                      0
PumpTimeInSeconds           0
WaterLevel                  0
hours_since_watering        0
soil_deficit             8306
soil_deficit_ratio       8306
temp_deviation           8461
air_hum_deficit          8607
light_deviation          9486
deficit_x_temp          16205
deficit_x_light         17093
deficit_x_air           16300
et_approx               24657
dtype: int64

In [8]:
missing_percent = df.isnull().mean() * 100
print(missing_percent)

mac_id                   0.000000
PumpTimeInSeconds        0.000000
WaterLevel               0.000000
hours_since_watering     0.000000
soil_deficit             6.898900
soil_deficit_ratio       6.898900
temp_deviation           7.027642
air_hum_deficit          7.148909
light_deviation          7.878999
deficit_x_temp          13.459749
deficit_x_light         14.197316
deficit_x_air           13.538656
et_approx               20.479916
dtype: float64


In [9]:
df['PumpTimeInSeconds'].mean()

np.float64(5.479774992524669)